# Детекция ботов (кейс Авито)

По событиям в суточном окне наблюдения строим модель, которая каждой `cookie_id`
из `test.csv` присваивает `score ∈ [0, 1]` — вероятность принадлежности куки
ботам (трафику сервисов автоматического сбора данных).

Официальная метрика — **Precision при Recall ≥ 0.70** — реализована
прямо в этом ноутбуке (функция `precision_at_recall`).

## О подходе

1. **Признаки строим только по событиям внутри окна наблюдения**:
   `window_start_ts ≤ event_ts < window_end_ts` — это требование условия.
2. **Идея признаков:** поток робота отличается от человеческого не только
   объемом, но и структурой: регулярные интервалы времени, повторные просмотры
   одних и тех же объявлений, отсутствие «мышиного» поведения (координаты
   курсора), круглосуточная активность, однотипные платформы и User-Agent.
3. **Валидация — строго хронологическая** (тест лежит позже train по времени,
   случайный сплит был бы некорректен): ранние дни train — обучение, последние
   4 дня — валидация.
4. **Модель** — LightGBM с регуляризацией + **бленд 21 модели** (7 seed ×
   250/300/400 деревьев, усреднение вероятностей) для устойчивости на
   скрытом тесте.
   Приемы ансамбля и регуляризации переняты из прошлого кейса
   `lead_prioritization_challenge`.
5. Все детерминировано: `random_state` зафиксирован, повторный запуск дает
   идентичный `submission.csv`.

## 0. Подготовка

Загружаем библиотеки, задаем конфигурацию, определяем путь к данным.

In [6]:
# 0. Подготовка
import os, re
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

TARGET_RECALL = 0.70


def pr_curve(y_true, score) -> tuple[np.ndarray, np.ndarray]:
    """Метрика кейса, часть 1: (precision, recall) в точках на границах групп.

    Куки сортируются по `score` по убыванию; одинаковые score обрабатываются
    одной группой (нельзя разделить куки с равным score).
    """
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if y_true.shape != score.shape:
        raise ValueError("y_true и score разной длины")

    n_pos = int(y_true.sum())
    if n_pos == 0:
        return np.array([]), np.array([])

    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]

    tp = np.cumsum(y)
    k = np.arange(1, len(y) + 1)
    ends = np.r_[s[1:] != s[:-1], True]      # последняя строка каждой группы равных score
    return tp[ends] / k[ends], tp[ends] / n_pos


def precision_at_recall(y_true, score, recall: float = TARGET_RECALL) -> float:
    """Метрика кейса: максимальный precision среди порогов с recall >= `recall`."""
    prec, rec = pr_curve(y_true, score)
    if len(prec) == 0:
        return float("nan")
    ok = rec >= recall
    return float(prec[ok].max()) if ok.any() else 0.0

# Конфигурация

def _resolve_data_dir() -> str:
    """Путь к данным: ищем папку с train.csv рядом с ноутбуком."""
    for cand in ("../data", "data"):
        if os.path.isfile(os.path.join(cand, "train.csv")):
            return cand
    raise FileNotFoundError("Не найден каталог с данными (train.csv)")

DATA_DIR = _resolve_data_dir()

RANDOM_STATE = 42                   # фиксируем для воспроизводимости
VAL_CUT = "2026-04-16"              # валидация: последние 4 дня train (тест идет позже)

MODEL_PARAMS = {
    "objective": "binary",
    "learning_rate": 0.03,
    "num_leaves": 127,            # подобрано по P@R на хроновалидации
    "min_child_samples": 60,      # регуляризация (небольшие листья режутся)
    "n_estimators": 250,          # 200-1000 деревьев: 250 оптимально на всех срезах (7 seed)
    "feature_fraction": 0.9,      # случайный отбор признаков на сплит
    "bagging_fraction": 0.9,      # случайный отбор строк на сплит
    "bagging_freq": 1,
    "reg_lambda": 1.0,            # L2-регуляризация против переобучения
    "random_state": RANDOM_STATE,
    "verbose": -1,
    "n_jobs": -1,
}

# Ансамбль: усредняем модели с разными seed — снижает разброс прогнозов
# на скрытом тесте при ограниченном числе попыток (7 seed стабильнее трех)
ENSEMBLE_SEEDS = [42, 123, 456, 7, 99, 2024, 555]

# Число деревьев на ногу ансамбля: смешиваем три набора деревьев — на всех
# хронологических срезах P@R бленда не ниже каждой ноги по отдельности
MODEL_N_TREES = [250, 300, 400]

EVENT_TYPES = [
    "item_view", "search_results_view", "photo_swipe", "favorite_add",
    "seller_page_view", "contact_phone_show", "contact_chat_open",
    "contact_message_sent", "login",
]

# Маркеры автоматизированного трафика в User-Agent
BOT_UA_REGEX = re.compile(
    r"bot|crawler|spider|scrapy|headless|python|requests|urllib|curl|fetch|"
    r"node|okhttp|httpclient",
    re.I,
)

CONTACT_EVENTS = ["contact_phone_show", "contact_message_sent", "contact_chat_open"]


def _entropy(counts: pd.Series) -> float:
    """Энтропия распределения: выше — равномернее набор действий/категорий."""
    p = counts / counts.sum()
    return float(-(p * np.log(p + 1e-9)).sum())
print("каталог данных:", os.path.abspath(DATA_DIR))


каталог данных: /Users/kamila/Desktop/bot_detection_challenge/data


## 1. Загрузка данных

`train.csv` / `test.csv` — метаданные куки и границы окна, `events.csv.gz` —
история событий. Сразу нормализуем технические поля: платформу, флаги
User-Agent, наличие координат курсора.

In [7]:
# 1. Загрузка данных

def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Загрузка train / test / events."""
    train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["cookie_created_at",
                                                              "window_start_ts",
                                                              "window_end_ts"])
    test = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=["cookie_created_at",
                                                            "window_start_ts",
                                                            "window_end_ts"])
    events = pd.read_csv(f"{DATA_DIR}/events.csv.gz", compression="gzip",
                         parse_dates=["event_ts"], low_memory=False)
    return train, test, events


def normalize_events(events: pd.DataFrame) -> pd.DataFrame:
    """
    Технические признаки, которые можно посчитать по всем событиям сразу:
    нормализованная платформа, флаги User-Agent, наличие координат курсора.
    """
    ua = events["user_agent"].fillna("").astype(str)

    def _norm_platform(p: object) -> str:
        p = str(p).strip().lower()
        if p in ("web", "desktop", "android"):
            return p
        if p in ("ios", "iphone"):
            return "ios"
        return "other"

    events = events.copy()
    events["platform_n"] = events["platform"].map(_norm_platform)
    events["ua_mobile"] = ua.str.contains("Mobile", case=False).astype(int)
    events["ua_android"] = ua.str.contains("Android", case=False).astype(int)
    events["ua_ios"] = ua.str.contains("iPhone|iPad|iOS", case=False).astype(int)
    events["ua_yab"] = ua.str.contains("YaBrowser", case=False).astype(int)
    events["pointer_ok"] = events["pointer_x"].notna().astype(int)
    return events
train, test, events = load_data()
events = normalize_events(events)
print(f"train: {train.shape}, test: {test.shape}, events: {events.shape}")
print(f"доля ботов в train: {train.target.mean():.4f}")
print("типы событий:", ", ".join(events.event_name.unique()))

train: (11091, 5), test: (4909, 4), events: (328905, 20)
доля ботов в train: 0.0811
типы событий: item_view, search_results_view, captcha_shown, photo_swipe, seller_page_view, favorite_add, contact_chat_open, contact_phone_show, login, contact_message_sent


## 2. Признаки по событиям в окне наблюдения

Группы признаков (`build_features`):

* **Объем и разнообразие** — число событий, уникальных объявлений/категорий/
  локаций/запросов, «дубли» просмотра (роботы переоткрывают одни и те же
  объявления), уникальность координат курсора.
* **Типы событий** — счетчики и доли (item_view, search, photo_swipe,
  seller_page, favorite, контактные действия — сигналы выгрузки контактов).
* **Временная структура** — интервалы между событиями (mean/median/std/min/p90,
  CV), регулярность (`top_gap_share` — самая частая длина интервала), часовая
  энтропия, ночная доля, пиковый час, всплески темпа за 10 секунд, число
  «сессий» (> 10 минут). Боты работают по расписанию и могут идти круглосуточно.
* **«Мышиное» поведение** — доля событий с координатами курсора, разброс
  координат: у роботов курсорной активности либо нет, либо координаты постоянны.
* **Платформа и User-Agent** — доли платформ, мобильности, Yandex Browser;
  число уникальных UA на куку, **доля событий с «ботовым» UA** (headless,
  python, scrapy, curl …).
* **Пагинация и поиск** — макс./средняя страница выдачи (`search_page`),
  «покрытие» страниц (`page_coverage`), повторяемость запросов, длина запроса.
* **Структура интересов** — энтропия типов событий и категорий объявлений,
  доля топ-категории; **активность на границах окна** (доля событий в первый/
  последний час), время до первого контакта, медианный интервал между
  просмотрами объявлений, макс. всплеск одного типа события за 10 секунд.
* **Тип продавца, возраст куки, повторные просмотры одних объявлений.**

In [8]:
def events_in_window(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """
    Оставляем только события внутри окна наблюдения:
    window_start_ts <= event_ts < window_end_ts (требование условия).
    Дубликаты строк (повторно записанные события) выбрасываем.
    """
    win = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]],
                       on="cookie_id", how="inner")
    win = win[(win["event_ts"] >= win["window_start_ts"]) &
              (win["event_ts"] < win["window_end_ts"])]
    win = win.drop(columns=["window_start_ts", "window_end_ts"])
    return win.sort_values(["cookie_id", "event_ts"]).drop_duplicates()


def build_features(meta: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """
    Признаки по истории событий внутри окна, агрегат на одну cookie_id.

    Идея: поток событий робота отличается от человеческого не только объемом,
    но и структурой — регулярные интервалы времени, повторяющиеся действия,
    отсутствие поведения курсора, однотипные платформы
    и User-Agent. Ниже каждая группа признаков закомментирована.
    """
    win = events_in_window(events, meta)
    if win.empty:
        cols = [c for c in _feature_template(meta).columns]
        return pd.DataFrame(0, index=meta.index, columns=cols)

    g = win.groupby("cookie_id")

    # базовый объем активности
    f = pd.DataFrame(index=meta["cookie_id"].values)
    n = g.size()
    f["n_events"] = n
    f["log_events"] = np.log1p(n)
    f["n_event_types"] = g["event_name"].nunique()   # разнообразие типов действий

    # разнообразие объектов, которыми интересуется кука
    f["n_unq_item"] = g["item_id"].nunique()
    f["n_unq_cat"] = g["item_category"].nunique()
    f["n_unq_loc"] = g["item_location"].nunique()
    f["n_unq_seller"] = g["seller_type"].nunique()
    f["n_unq_query"] = g["search_query"].nunique()
    f["n_unq_page"] = g["search_page"].nunique()
    f["n_unq_px"] = g["pointer_x"].nunique()
    f["n_unq_py"] = g["pointer_y"].nunique()
    f["n_unq_ua"] = g["user_agent"].nunique()
    f["n_unq_plat"] = g["platform_n"].nunique()
    # "дубли" просмотра: роботы часто переоткрывают одни и те же объявления
    f["item_dupl"] = n / (f["n_unq_item"] + 1) - 1
    f["cat_dupl"] = n / (f["n_unq_cat"] + 1) - 1
    f["items_per_search"] = f["n_unq_item"] / (f["n_unq_query"] + 1)
    f["loc_per_query"] = f["n_unq_loc"] / (f["n_unq_query"] + 1)

    # счетчики по типам событий и их доли
    evc = pd.crosstab(win["cookie_id"], win["event_name"]).reindex(f.index, fill_value=0)
    evc.columns = [f"ev_{c}" for c in evc.columns]
    f = f.join(evc)
    for c in EVENT_TYPES:
        f[f"sh_ev_{c}"] = f.get(f"ev_{c}", 0) / (f["n_events"] + 1)
    f["contact_act"] = sum(f.get(f"ev_{c}", 0) for c in
                           ["contact_phone_show", "contact_message_sent", "contact_chat_open"])

    # временная структура потока событий
    # Боты работают по расписанию: интервалы между событиями регулярнее,
    # активность может идти круглосуточно, а не в бодрствующем окне человека
    first = g["event_ts"].min()
    last = g["event_ts"].max()
    f["span_h"] = (last - first).dt.total_seconds().div(3600).fillna(0)
    f["ev_per_h"] = f["n_events"] / (f["span_h"] + 1e-9)

    dif = win.groupby("cookie_id")["event_ts"].diff().dt.total_seconds().dropna()
    gap = dif.to_frame("gap")
    gap["cookie_id"] = win.loc[dif.index, "cookie_id"].values
    dg = gap.groupby("cookie_id")["gap"]
    f["gap_mean"] = dg.mean()
    f["gap_median"] = dg.median()
    f["gap_std"] = dg.std()
    f["gap_min"] = dg.min()
    f["gap_p90"] = dg.quantile(0.9)
    f["gap_cv"] = dg.std() / (dg.mean() + 1e-9)

    # регулярность: доля наиболее частого "округляемого" интервала
    bucket = (dif / (dif.std() + 1e-9) * 1e5).round().to_frame("b")
    bucket["cookie_id"] = win.loc[dif.index, "cookie_id"].values
    f["top_gap_share"] = bucket.groupby("cookie_id")["b"].apply(
        lambda s: s.value_counts(normalize=True).iloc[0] if len(s) else 0.0)
    f["n_distinct_gaps"] = dg.nunique()

    # расписание по часам суток: энтропия активности, ночная доля, пиковый час
    hours = win["event_ts"].dt.hour
    hcnt = pd.crosstab(win["cookie_id"], hours).reindex(f.index, fill_value=0)
    p_h = hcnt.div(hcnt.sum(axis=1), axis=0).fillna(0)
    f["hour_entropy"] = -(p_h * np.log(p_h + 1e-9)).sum(axis=1)
    f["night_frac"] = hcnt[[0, 1, 2, 3, 4, 5]].sum(axis=1) / (hcnt.sum(axis=1) + 1)
    f["n_active_hours"] = (hcnt > 0).sum(axis=1)
    f["peak_hour"] = hcnt.idxmax(axis=1)

    # всплески темпов: робот может делать много запросов в секунду
    sec10 = win["event_ts"].astype("int64") // (10 * 10 ** 9)
    min10 = pd.crosstab(win["cookie_id"], sec10).reindex(f.index, fill_value=0)
    f["max_per_10s"] = min10.max(axis=1)
    f["n_active_10s_bins"] = (min10 > 0).sum(axis=1)
    f["cpm_burst"] = f["max_per_10s"] * 6

    # число "сессий" — разрывов дольше 10 минут
    sess = (dif > 600).astype(int).to_frame("s")
    sess["cookie_id"] = win.loc[dif.index, "cookie_id"].values
    f["n_sessions"] = sess.groupby("cookie_id")["s"].sum()

    # "мышиное" поведение: координаты курсора
    # У роботов координаты курсора либо отсутствуют, либо постоянны
    f["pointer_frac"] = win.groupby("cookie_id")["pointer_ok"].mean()
    f["ptr_itemview"] = win[win["event_name"] == "item_view"] \
        .groupby("cookie_id")["pointer_ok"].mean()
    f["px_mean"] = win.groupby("cookie_id")["pointer_x"].mean()
    f["py_mean"] = win.groupby("cookie_id")["pointer_y"].mean()
    f["px_std"] = win.groupby("cookie_id")["pointer_x"].std()
    f["py_std"] = win.groupby("cookie_id")["pointer_y"].std()

    # платформы и User-Agent
    plat = pd.crosstab(win["cookie_id"], win["platform_n"]).reindex(f.index, fill_value=0)
    for c in plat.columns:
        f[f"plat_{c}"] = plat[c] / (f["n_events"] + 1)
    f["ua_mobile_frac"] = win.groupby("cookie_id")["ua_mobile"].mean()
    f["ua_android_frac"] = win.groupby("cookie_id")["ua_android"].mean()
    f["ua_ios_frac"] = win.groupby("cookie_id")["ua_ios"].mean()
    f["ua_yab_frac"] = win.groupby("cookie_id")["ua_yab"].mean()

    # тип продавца
    seller = pd.crosstab(win["cookie_id"], win["seller_type"].fillna("none"))
    seller = seller.reindex(f.index, fill_value=0)
    for c in seller.columns:
        f[f"seller_{c}"] = seller[c] / (f["n_events"] + 1)

    # возраст куки (метаданные)
    age_days = (meta["window_start_ts"] - meta["cookie_created_at"]).dt.total_seconds().div(86400)
    f["cookie_age_d"] = age_days.clip(lower=0)

    # многократные просмотры одних объявлений
    item_cnt = win.groupby(["cookie_id", "item_id"]).size()
    f["max_item_cnt"] = item_cnt.groupby(level=0).max().reindex(f.index, fill_value=0)
    top2 = item_cnt.groupby(level=0).apply(lambda s: s.nlargest(2).sum())
    f["top2_item_share"] = top2.reindex(f.index, fill_value=0) / f["n_unq_item"].clip(lower=1)

    # повторяемость просмотров одних и тех же объявлений
    iv = win[win["event_name"] == "item_view"]
    _uiv = iv.groupby("cookie_id")["item_id"].nunique()
    _viv = iv.groupby("cookie_id")["item_id"].size()
    f["view_repeat"] = (_viv - _uiv) / (_uiv + 1)

    # User-Agent с признаками автоматизации
    # Headless-браузеры, скрейперы и http-клиенты прямо заявляют себя в UA
    _ua_b = win["user_agent"].fillna("").astype(str).str.contains(BOT_UA_REGEX).astype(int)
    f["ua_bot_frac"] = _ua_b.groupby(win["cookie_id"]).mean()
    f["ua_bot_any"] = _ua_b.groupby(win["cookie_id"]).max()

    # пагинация выдачи
    # Роботы перебирают страницы выдачи последовательно и нередко глубоко
    _pg = win["search_page"]
    f["search_page_max"] = _pg.groupby(win["cookie_id"]).max()
    f["search_page_mean"] = _pg.groupby(win["cookie_id"]).mean().fillna(0)
    f["search_page_gt1"] = (_pg > 1).astype(int).groupby(win["cookie_id"]).mean()
    _spn = win[_pg.notna()].groupby("cookie_id")["search_page"].nunique()
    f["page_coverage"] = _spn / (f["search_page_max"].fillna(0) + 1)

    # поисковые запросы
    _se = win[win["event_name"] == "search_results_view"]
    _ns = _se.groupby("cookie_id").size()
    _qn = _se.groupby("cookie_id")["search_query"].nunique()
    f["query_repeat"] = _ns / (_qn + 1)
    f["avg_query_len"] = _se.groupby("cookie_id")["search_query"].apply(
        lambda s: s.dropna().astype(str).str.len().mean())

    # энтропии: равномерность структуры интересов
    # У роботов поток событий и категорий часто ровнее, чем у человека
    _ec = pd.crosstab(win["cookie_id"], win["event_name"])
    f["seq_entropy"] = _ec.apply(_entropy, axis=1)
    _cc = pd.crosstab(win["cookie_id"], win["item_category"].fillna("none"))
    f["cat_entropy"] = _cc.apply(_entropy, axis=1)
    f["top_cat_share"] = _cc.div(_cc.sum(axis=1), axis=0).max(axis=1)

    # активность на границах окна
    # Кука, начавшая работать синхронно со стартом окна или не умолкающая
    # до его конца, похожа на планировщик, а не на человека
    win["sec_from_first"] = (win["event_ts"] - win.groupby("cookie_id")["event_ts"].transform("min")).dt.total_seconds()
    f["first_hour_frac"] = (win["sec_from_first"] < 3600).astype(int).groupby(win["cookie_id"]).mean()
    _last_map = f["span_h"] * 3600
    f["last_hour_frac"] = ((win["sec_from_first"] + 3600) > win["cookie_id"].map(_last_map)).astype(int).groupby(win["cookie_id"]).mean()

    # связь просмотр -> контакт
    # Бот собирает контакты без обдумывания: от первого шага до контакта проходит
    # мало времени. Нет контакта — заполняем заведомо большим значением
    _cont = win[win["event_name"].isin(CONTACT_EVENTS)]
    f["min_till_contact_h"] = f["span_h"] + 12
    if len(_cont):
        _mt = _cont.groupby("cookie_id")["sec_from_first"].min() / 3600
        f.loc[_mt.index, "min_till_contact_h"] = _mt.values

    # регулярность просмотров объявлений
    # У роботов интервалы между переоткрытиями карточек стабильны
    if len(iv):
        _vi = iv.groupby("cookie_id")["event_ts"].diff().dt.total_seconds().dropna()
        _vg = _vi.to_frame("v").assign(cid=iv.loc[_vi.index, "cookie_id"].values)
        f["itemview_gap_med"] = _vg.groupby("cid")["v"].median()

    # максимальный всплеск одного типа события за 10 секунд
    _step = win.assign(bucket=win["event_ts"].astype("int64") // (10 * 10 ** 9)) \
        .groupby(["cookie_id", "bucket", "event_name"]).size().rename("cnt").reset_index()
    f["max_same_type_10s"] = _step.groupby("cookie_id")["cnt"].max()

    # время жизни куки и старт активности
    # Кука, созданная прямо внутри окна или проснувшаяся одновременно со стартом
    # окна, — типичный признак планировщика/робота (у людей обычно есть история)
    _delay = ((win["event_ts"] - win["cookie_id"].map(meta.set_index("cookie_id")["cookie_created_at"]))
              .dt.total_seconds())
    f["first_event_delay_s"] = _delay.groupby(win["cookie_id"]).min()
    f["created_in_window"] = ((meta["cookie_created_at"] >= meta["window_start_ts"]) &
                              (meta["cookie_created_at"] <= meta["window_end_ts"])).astype(int)

    # переходы между типами событий (2-граммы)
    # Структура последовательности: человек чередует просмотр с фото/фаворитами,
    # робот линейно листает выдачу -> просмотр -> просмотр -> ...
    _prev = g["event_name"].shift()
    _tr = win.assign(prev=_prev)[_prev.notna()]
    _tr["trans"] = _tr["prev"] + "|" + _tr["event_name"]
    _st = _tr.groupby("cookie_id")["trans"]
    _cnt = _st.size()
    f["top_trans_share"] = _st.apply(
        lambda s: s.value_counts(normalize=True).iloc[0] if len(s) else 0.0)
    f["trans_entropy"] = _st.apply(lambda s: 0.0 if len(s) == 0 else _entropy(s.value_counts()))
    f["n_trans"] = _cnt
    for _tag, _pat in [("sv", "search_results_view|item_view"),
                       ("vv", "item_view|item_view")]:
        f[f"trans_{_tag}_frac"] = _tr[_tr["trans"] == _pat].groupby("cookie_id").size() / (_cnt + 1)

    # выравниваем по порядку строк метаданных
    f = f.reindex(meta["cookie_id"].values)
    f.index = meta.index
    return f


def _feature_template(meta: pd.DataFrame) -> pd.DataFrame:
    """Заглушка структуры признаков (используется при пустом окне событий)."""
    return pd.DataFrame(index=meta.index)


def fill_na(df: pd.DataFrame) -> pd.DataFrame:
    """Пропуски в числовых признаках (кука без событий в окне) -> 0."""
    num = df.select_dtypes(include=[np.number]).columns
    df[num] = df[num].fillna(0)
    return df


def union_columns(train_feat: pd.DataFrame, test_feat: pd.DataFrame):
    """Согласовываем набор колонок train/test (одинаковые признаки)."""
    cols = list(train_feat.columns) + [c for c in test_feat.columns
                                       if c not in train_feat.columns]
    train_feat = train_feat.reindex(columns=cols, fill_value=0)
    test_feat = test_feat.reindex(columns=cols, fill_value=0)
    return train_feat, test_feat

## 3. Валидация (хронологическая) и обучение модели

Тест лежит **после** train по времени, поэтому обучаемся на ранних днях train,
а на последних 4 днях честно оцениваем метрику кейса встроенной функцией
`precision_at_recall` (группы равных score обрабатываются целиком).

Затем обучаем **бленд из 21 модели (7 seed × 250/300/400 деревьев)**:
каждая модель — на «ранних» днях (оценка валидации) и на всех данных train
(финальный прогноз на тест). Прогнозы усредняются.

In [9]:
# 3. Строим признаки, валидируем и обучаем ансамбль
X_train = fill_na(build_features(train, events))
X_test = fill_na(build_features(test, events))
X_train, X_test = union_columns(X_train, X_test)
y = train["target"].values
print(f"признаков: {len(X_train.columns)}")

# Хронологическое разделение: ранние дни train -> обучение, последние 4 дня -> валидация
is_val = train["window_start_ts"] >= pd.Timestamp(VAL_CUT)
is_train = ~is_val

# Число деревьев варьируется по ногам ансамбля (MODEL_N_TREES):
# смешивание 250/300/400 деревьев на всех хрон. срезах не хуже каждой ноги
val_probs, test_probs = [], []
for n_trees in MODEL_N_TREES:
    for seed in ENSEMBLE_SEEDS:
        params = {**MODEL_PARAMS, "n_estimators": n_trees, "random_state": seed}
        model = lgb.LGBMClassifier(**params)
        # модель на ранних днях -> прогноз на валидации
        model.fit(X_train.loc[is_train], y[is_train])
        val_probs.append(model.predict_proba(X_train.loc[is_val])[:, 1])
        # финальная модель на ВСЕХ данных train -> прогноз на тесте
        model.fit(X_train, y)
        test_probs.append(model.predict_proba(X_test)[:, 1])

p_val = np.mean(val_probs, axis=0)
p_test = np.mean(test_probs, axis=0)

print("\nВалидация (невидимые при обучении дни)")
print(f"дней обучения: {is_train.sum()}, дней валидации: {is_val.sum()}")
print(f"доля ботов на валидации (уровень константы):    {y[is_val].mean():.4f}")
print(f"P@R>=0.7 на валидации:                           {precision_at_recall(y[is_val], p_val):.4f}")
print(f"ROC-AUC на валидации:                            {roc_auc_score(y[is_val], p_val):.4f}")
print(f"ансамбль из {len(MODEL_N_TREES) * len(ENSEMBLE_SEEDS)} моделей ({len(MODEL_N_TREES)} набора деревьев x {len(ENSEMBLE_SEEDS)} seed)")

признаков: 98

Валидация (невидимые при обучении дни)
дней обучения: 8400, дней валидации: 2691
доля ботов на валидации (уровень константы):    0.0818
P@R>=0.7 на валидации:                           0.7230
ROC-AUC на валидации:                            0.9296
ансамбль из 21 моделей (3 набора деревьев x 7 seed)


## 4. Сабмит

Формируем `submission.csv` (ровно две колонки: `cookie_id`, `score`);
проверяем формат: число строк совпадает с `test.csv`, дубликатов кук нет,
score лежит в `[0, 1]`.

In [10]:
# 4. Сабмит
submission = pd.DataFrame({"cookie_id": test["cookie_id"].values, "score": p_test})
assert len(submission) == len(test), "число строк не совпадает с test.csv"
assert submission["cookie_id"].is_unique, "есть дубликаты cookie_id"
assert submission["score"].between(0, 1).all(), "score вне [0,1]"
submission.to_csv("submission.csv", index=False)
print(f"\nsubmission.csv сохранен: {len(submission)} строк")
submission.head()


submission.csv сохранен: 4909 строк


,cookie_id,score
0,ck_315fb710a0e371e7,0.000624
1,ck_a76ee3b3e3e522fd,0.045772
2,ck_94c9a4d382689e82,0.018827
3,ck_8eaf9509ad9462a0,0.000654
4,ck_9a88a5a989cb5bc6,0.001816
